In [1]:
# Cell 1: Install Required Libraries
# Run this first to install all necessary packages
!pip install requests beautifulsoup4 pandas lxml

   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   -- ------------------------------------- 0.3/4.0 MB ? eta -:--:--
   -- ------------------------------------- 0.3/4.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/4.0 MB 533.6 kB/s eta 0:00:07
   ----- ---------------------------------- 0.5/4.0 MB 533.6 kB/s eta 0:00:07
   ----- ---------------------------------- 0.5/4.0 MB 533.6 kB/s eta 0:00:07
   ----- ---------------------------------- 0.5/4.0 MB 533.6 kB/s eta 0:00:07
   ----- ---------------------------------- 0.5/4.0 MB 533.6 kB/s eta 0:00:07
   ----- ---------------------------------- 0.5/4.0 MB 533.6 kB/s eta 0:00:07
   ----- ---------------------------------- 0.5/4.0 MB 533.6 kB/s eta 0:00:07
   ------- -------------------------------- 0.8/4.0 MB 259.8 kB/s eta 0:00:13
   ------- --------------------


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Cell 2: Import Libraries
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from typing import Dict, Optional
import re

In [3]:
# Cell 3: Define Scraping Functions
def get_chapter_data(chapter_number: int) -> Dict[str, Optional[str]]:
    """
    Scrape data from a single One Piece chapter.
    
    Args:
        chapter_number: The chapter number to scrape
        
    Returns:
        Dictionary containing chapter_number, cover_page, and in_depth_summary
    """
    url = f"https://onepiece.fandom.com/wiki/Chapter_{chapter_number}"
    
    try:
        # Add headers to mimic a browser request
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
        
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Initialize return dictionary
        data = {
            'chapter_number': chapter_number,
            'cover_page': None,
            'in_depth_summary': None
        }
        
        # Find all h2 headers to locate sections
        headers = soup.find_all(['h2', 'span'], class_=['mw-headline'])
        
        # Extract Cover Page
        for header in headers:
            header_text = header.get_text().strip()
            
            # Look for Cover Page section
            if 'Cover Page' in header_text or 'Color Spread' in header_text:
                # Find the next sibling elements until we hit another h2
                cover_content = []
                current = header.parent.find_next_sibling()
                
                while current and current.name != 'h2':
                    if current.name in ['p', 'ul', 'dl']:
                        text = current.get_text().strip()
                        if text:
                            cover_content.append(text)
                    current = current.find_next_sibling()
                
                data['cover_page'] = '\n'.join(cover_content) if cover_content else None
            
            # Look for Long Summary (In-depth Summary)
            elif 'Long Summary' in header_text:
                # Find the next sibling elements until we hit another h2
                summary_content = []
                current = header.parent.find_next_sibling()
                
                while current and current.name != 'h2':
                    if current.name in ['p']:
                        text = current.get_text().strip()
                        if text:
                            summary_content.append(text)
                    current = current.find_next_sibling()
                
                data['in_depth_summary'] = '\n\n'.join(summary_content) if summary_content else None
        
        return data
        
    except requests.exceptions.RequestException as e:
        print(f"Error fetching chapter {chapter_number}: {e}")
        return {
            'chapter_number': chapter_number,
            'cover_page': None,
            'in_depth_summary': None,
            'error': str(e)
        }
    except Exception as e:
        print(f"Error parsing chapter {chapter_number}: {e}")
        return {
            'chapter_number': chapter_number,
            'cover_page': None,
            'in_depth_summary': None,
            'error': str(e)
        }

In [4]:
# Cell 4: Scrape All Chapters with Progress Tracking
def scrape_all_chapters(start_chapter: int = 1, end_chapter: int = 1162, 
                        delay: float = 1.0, save_frequency: int = 50):
    """
    Scrape all chapters from start_chapter to end_chapter.
    
    Args:
        start_chapter: First chapter to scrape (default: 1)
        end_chapter: Last chapter to scrape (default: 1162)
        delay: Delay in seconds between requests (default: 1.0)
        save_frequency: Save progress every N chapters (default: 50)
        
    Returns:
        DataFrame containing all scraped data
    """
    all_data = []
    
    print(f"Starting to scrape chapters {start_chapter} to {end_chapter}")
    print(f"This will take approximately {(end_chapter - start_chapter + 1) * delay / 60:.1f} minutes")
    print("-" * 60)
    
    for chapter_num in range(start_chapter, end_chapter + 1):
        print(f"Scraping Chapter {chapter_num}/{end_chapter}...", end=' ')
        
        # Scrape the chapter
        chapter_data = get_chapter_data(chapter_num)
        all_data.append(chapter_data)
        
        # Print status
        if 'error' in chapter_data:
            print("❌ ERROR")
        elif chapter_data['cover_page'] or chapter_data['in_depth_summary']:
            print("✓")
        else:
            print("⚠ No data found")
        
        # Save intermediate results
        if chapter_num % save_frequency == 0:
            temp_df = pd.DataFrame(all_data)
            temp_df.to_csv(f'onepiece_chapters_backup_{chapter_num}.csv', index=False)
            print(f"  → Backup saved at chapter {chapter_num}")
        
        # Be respectful to the server - add delay between requests
        if chapter_num < end_chapter:
            time.sleep(delay)
    
    print("-" * 60)
    print("✓ Scraping complete!")
    
    # Convert to DataFrame
    df = pd.DataFrame(all_data)
    return df

In [5]:
# Cell 5: Run the Scraper
# Adjust parameters as needed:
# - start_chapter: where to begin (default: 1)
# - end_chapter: where to end (default: 1162)
# - delay: seconds between requests (default: 1.0 - be respectful!)
# - save_frequency: save backup every N chapters (default: 50)

df = scrape_all_chapters(
    start_chapter=1,
    end_chapter=1162,
    delay=1.0,
    save_frequency=50
)

Starting to scrape chapters 1 to 1162
This will take approximately 19.4 minutes
------------------------------------------------------------
Scraping Chapter 1/1162... ✓
Scraping Chapter 2/1162... ✓
Scraping Chapter 3/1162... ✓
Scraping Chapter 4/1162... ✓
Scraping Chapter 5/1162... ✓
Scraping Chapter 6/1162... ✓
Scraping Chapter 7/1162... ✓
Scraping Chapter 8/1162... ✓
Scraping Chapter 9/1162... ✓
Scraping Chapter 10/1162... ✓
Scraping Chapter 11/1162... ✓
Scraping Chapter 12/1162... ✓
Scraping Chapter 13/1162... ✓
Scraping Chapter 14/1162... ✓
Scraping Chapter 15/1162... ✓
Scraping Chapter 16/1162... ✓
Scraping Chapter 17/1162... ✓
Scraping Chapter 18/1162... ✓
Scraping Chapter 19/1162... ✓
Scraping Chapter 20/1162... ✓
Scraping Chapter 21/1162... ✓
Scraping Chapter 22/1162... ✓
Scraping Chapter 23/1162... ✓
Scraping Chapter 24/1162... ✓
Scraping Chapter 25/1162... ✓
Scraping Chapter 26/1162... ✓
Scraping Chapter 27/1162... ✓
Scraping Chapter 28/1162... ✓
Scraping Chapter 29/1162... 

In [6]:
# Cell 6: Save to CSV
output_filename = 'onepiece_chapters_complete.csv'
df.to_csv(output_filename, index=False)
print(f"✓ Data saved to {output_filename}")

✓ Data saved to onepiece_chapters_complete.csv


In [7]:
# Cell 7: Display Statistics
print("=" * 60)
print("SCRAPING STATISTICS")
print("=" * 60)
print(f"Total chapters processed: {len(df)}")
print(f"Chapters with cover page: {df['cover_page'].notna().sum()}")
print(f"Chapters with in-depth summary: {df['in_depth_summary'].notna().sum()}")
print(f"Chapters with errors: {df.get('error', pd.Series()).notna().sum()}")
print("=" * 60)

SCRAPING STATISTICS
Total chapters processed: 1162
Chapters with cover page: 1148
Chapters with in-depth summary: 1158
Chapters with errors: 4


In [8]:
# Cell 8: View Sample Data
# Display first few rows
print("\nFirst 3 chapters:")
print(df[['chapter_number', 'cover_page', 'in_depth_summary']].head(3))



First 3 chapters:
   chapter_number                                         cover_page  \
0               1  Color Spread: Luffy, Nami, and the Red Hair Pi...   
1               2  Animal Theater: Luffy stands among a flock of ...   
2               3  Animal Theater: Luffy rides a fierce polka-dot...   

                                    in_depth_summary  
0  A man is handcuffed and in front of him there ...  
1  Luffy sets off in a dinghy to form his own pir...  
2  Out at sea, Koby and Luffy are discussing Luff...  


In [12]:
# Cell 9: Export to Excel (Optional)
!pip install openpyxl

df.to_excel('onepiece_chapters_complete.xlsx', index=False)
print("✓ Data also saved to Excel format")



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------

In [ ]:
# Cell 10: Resume from Backup (if needed)
# If scraping was interrupted, use this to resume
def resume_scraping(backup_file: str, end_chapter: int = 1162):
    """
    Resume scraping from a backup file.
    
    Args:
        backup_file: Path to the backup CSV file
        end_chapter: Last chapter to scrape
    """
    # Load existing data
    df_existing = pd.read_csv(backup_file)
    last_chapter = df_existing['chapter_number'].max()
    
    print(f"Resuming from chapter {last_chapter + 1}")
    
    # Scrape remaining chapters
    df_new = scrape_all_chapters(
        start_chapter=last_chapter + 1,
        end_chapter=end_chapter,
        delay=1.0,
        save_frequency=50
    )
    
    # Combine data
    df_combined = pd.concat([df_existing, df_new], ignore_index=True)
    return df_combined

# Example usage (uncomment if needed):
# df = resume_scraping('onepiece_chapters_backup_100.csv', end_chapter=1162)